# Kaggle x2 —  chạy đồng thời hai GPU

In [ ]:
from pathlib import Path
import os

# Dùng repo đã clone/upload; chọn đường dẫn phù hợp phiên làm việc.
REPO_DIR = Path("/kaggle/working/code")

if Path("train.py").exists():
    REPO_DIR = Path.cwd()

REPOSITORY_URL = "https://github.com/lngphgthao/moe-vihsd.git"
BRANCH = "main" 

if not REPO_DIR.exists():
    !git clone --depth 1 --branch "$BRANCH" "$REPOSITORY_URL" "$REPO_DIR"
    
assert (REPO_DIR / "train.py").is_file(), "Đặt REPO_DIR đến checkout/upload chứa code mới"
os.chdir(REPO_DIR)

In [ ]:
!python -m pip install -q -r requirements.txt
assert _exit_code == 0, "Cài dependency thất bại"

In [ ]:
## Check GPU availability
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

!python -c "import torch; assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Notebook cần đúng hai GPU khả dụng'"
assert _exit_code == 0, "Chọn GPU T4 x2 trong Kaggle Settings trước khi chạy"

## Secrets

Smoke tắt W&B ở cả hai option; full dùng W&B online theo config và cần `WANDB_API_KEY`.
Thêm key vào Secrets của nền tảng; `HF_TOKEN` là tùy chọn.


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print(f"Chưa lấy được secret {key}; thêm vào Kaggle Secrets nếu cần")
        
os.environ["RUNS_DIR"] = "/kaggle/working/runs"

## Smoke — chạy đúng một option

Bỏ comment lệnh và assert của **A** (hai config riêng) hoặc **B** (`run_group`),
giữ option còn lại comment. Smoke dùng 1 epoch/64 train/32 validation, W&B tắt;
không dùng metric smoke để chọn model. Không Run All trước khi chọn option.

`shlex.quote(...)` giúp truyền cả đoạn shell an toàn thành một argument cho `bash -c`.
- Với `bash -c SCRIPT _ "python" "{OUTPUT_DIR}"`, `_` là `$0`, `python` là `$1` (lệnh Python), còn `{OUTPUT_DIR}` là `$2` (thư mục group YAML runtime).
- `CUDA_VISIBLE_DEVICES=0`/`1` chọn GPU cho từng process. `&` chạy process nền;
- `$!` lưu PID. `wait` chờ process đó và `$?` là mã thoát (0 nghĩa là thành công).
- Lệnh cuối dùng `&&` để cell thất bại nếu một trong hai process lỗi.


In [ ]:
TRAIN_PAIR_COMMAND = r"""
# Run each dataset on a different GPU, in the background.
CUDA_VISIBLE_DEVICES=0 "$1" train.py \
  --config  \
  --run-id run-020 \
  --smoke-test \
  --set logging.use_wandb=false &
first_pid=$!

CUDA_VISIBLE_DEVICES=1 "$1" train.py \
  --config configs/experiments/vihsd/exp-011.yaml \
  --run-id run-021 \
  --smoke-test \
  --set logging.use_wandb=false &
second_pid=$!

# Wait for both processes, even if the first one fails.
wait "$first_pid"
first_status=$?
wait "$second_pid"
second_status=$?

# Return failure if either process failed.
[ "$first_status" -eq 0 ] && [ "$second_status" -eq 0 ]
"""

In [ ]:
GROUP_PAIR_COMMAND = r"""
# Here, $2 is the folder containing the generated group YAML files.
CUDA_VISIBLE_DEVICES=0 "$1" scripts/run_group.py \
  --group-config "$2/vianli_smoke_group.yaml" \
  --auto-run-ids \
  --smoke-test &
first_pid=$!

CUDA_VISIBLE_DEVICES=1 "$1" scripts/run_group.py \
  --group-config "$2/vihsd_smoke_group.yaml" \
  --auto-run-ids \
  --smoke-test &
second_pid=$!

wait "$first_pid"
first_status=$?
wait "$second_pid"
second_status=$?

[ "$first_status" -eq 0 ] && [ "$second_status" -eq 0 ]
"""

In [ ]:
import shlex

# Shell chờ cả hai worker; một worker lỗi thì cell thất bại, vẫn giữ log/artifact.
# OPTION A — hai train process đồng thời:
# !bash -c {shlex.quote(TRAIN_PAIR_COMMAND)} _ "python"
# assert _exit_code == 0, "Một hoặc cả hai smoke worker thất bại; xem console.log từng run"

In [ ]:
# OPTION B — hai run_group process đồng thời, mỗi group một config:
# !bash -c {shlex.quote(GROUP_PAIR_COMMAND)} _ "python" "{OUTPUT_DIR}"
# assert _exit_code == 0, "Một hoặc cả hai smoke group thất bại; xem console.log từng run"

## Full baseline — chỉ sau khi cả hai smoke đạt

Đối chiếu status, loss hữu hạn và checkpoint của cả hai smoke trước. Chỉ bật một option.
Hai config chạy đồng thời, seed 42, ID run-022/025 từ group chính. Nếu ID đã dùng,
sửa group chính và lệnh A, chạy lại cell tạo group runtime trước khi launch.
Log trực tiếp có thể xen kẽ; từng run vẫn giữ `console.log` riêng trong `RUNS_DIR`.
Không gộp VRAM của hai GPU. Hai notebook main chạy tuần tự trên một GPU.


In [ ]:
import shlex

TRAIN_PAIR_COMMAND = r"""
# Run each dataset on a different GPU, in the background.
CUDA_VISIBLE_DEVICES=0 "$1" train.py \
  --config configs/experiments/vianli/exp-010.yaml \
  --run-id run-022 \
  --no-smoke-test &
first_pid=$!

CUDA_VISIBLE_DEVICES=1 "$1" train.py \
  --config configs/experiments/vihsd/exp-011.yaml \
  --run-id run-025 \
  --no-smoke-test &
second_pid=$!

# Wait for both processes, even if the first one fails.
wait "$first_pid"
first_status=$?
wait "$second_pid"
second_status=$?

# Return failure if either process failed.
[ "$first_status" -eq 0 ] && [ "$second_status" -eq 0 ]
"""

GROUP_PAIR_COMMAND = r"""
# Here, $2 is the folder containing the generated group YAML files.
CUDA_VISIBLE_DEVICES=0 "$1" scripts/run_group.py \
  --group-config "$2/vianli_full_group.yaml" \
  --auto-run-ids \
  --no-smoke-test &
first_pid=$!

CUDA_VISIBLE_DEVICES=1 "$1" scripts/run_group.py \
  --group-config "$2/vihsd_full_group.yaml" \
  --auto-run-ids \
  --no-smoke-test &
second_pid=$!

wait "$first_pid"
first_status=$?
wait "$second_pid"
second_status=$?

[ "$first_status" -eq 0 ] && [ "$second_status" -eq 0 ]
"""


In [ ]:
assert os.environ.get("WANDB_API_KEY"), "Thêm WANDB_API_KEY trước full run"

In [ ]:
# Shell chờ cả hai worker; một worker lỗi thì cell thất bại, vẫn giữ log/artifact.
# OPTION A — hai train process đồng thời:
# !bash -c {shlex.quote(TRAIN_PAIR_COMMAND)} _ "python"
# assert _exit_code == 0, "Một hoặc cả hai full worker thất bại; xem console.log từng run"

In [ ]:
# OPTION B — hai run_group process đồng thời, mỗi group một config:
# !bash -c {shlex.quote(GROUP_PAIR_COMMAND)} _ "python"
# assert _exit_code == 0, "Một hoặc cả hai full group thất bại; xem console.log từng run"

## Aggregate và lưu tracker

Kết quả aggregate nằm dưới `/kaggle/working/runs/analysis`; giữ cả run directories và
tracker CSV khi Save Version hoặc tải output.

In [ ]:
!python scripts/aggregate_results.py --runs-dir /kaggle/working/runs --output-dir /kaggle/working/runs/analysis
assert _exit_code == 0, "Tổng hợp kết quả thất bại"

!cp experiments/experiment_tracker.csv /kaggle/working/experiment_tracker.csv
assert _exit_code == 0, "Lưu tracker thất bại"

## Test set — chỉ sau khi khóa finalist

Không dùng test cho baseline phát triển. Chỉ bỏ comment sau khi config finalist đã khóa.

In [ ]:
# !python evaluate.py --run-dir "/kaggle/working/runs/<run-id>" --finalist-config "configs/finalists/<dataset>/<experiment-id>.yaml" --final-evaluation
# assert _exit_code == 0, "Final evaluation thất bại; giữ log và artifact"